# Movement Strategy Simulation

This notebook simulates each movement strategy with the same `MotorController` setup used in `backend.py`, then analyzes:
- each motor command over time
- reconstructed tactor position from resulting motor outputs (not requested path)
- wire consistency issues (too tight / inconsistent trilateration residual)

Default pattern (fully configurable):
1. Start at `(0, 0)`
2. Move right from center to `50%` of half-width
3. One clockwise circle
4. Return to `(0, 0)`


In [ ]:
import math
from dataclasses import dataclass
from pathlib import Path
import sys
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import clear_output, display

# Make imports work whether the notebook is launched from the repo root or from analysis/.
# Do not add any path outside this repository.
CURRENT_DIR = Path.cwd().resolve()
REPO_ROOT = CURRENT_DIR if (CURRENT_DIR / 'motor_controller.py').exists() else CURRENT_DIR.parent
if not (REPO_ROOT / 'motor_controller.py').exists():
    raise FileNotFoundError(
        'Could not locate the repository root. Launch this notebook from the repo root '
        'or from analysis/.'
    )

repo_root_str = str(REPO_ROOT)
if repo_root_str not in sys.path:
    sys.path.insert(0, repo_root_str)

from consts import EDGE_THRESHOLD, TOP_HEIGHT, TOP_WIDTH
from haptic_mapping import map_object_displacement_to_tactor
from motor_controller import HandOrientation, MotorController, MotorSetId, MovementStrategy

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.grid'] = True


In [ ]:
@dataclass
class SimulationConfig:
    # MotorController setup (mirrors backend usage)
    top_width: float = TOP_WIDTH
    top_height: float = TOP_HEIGHT
    edge_threshold: float = EDGE_THRESHOLD
    motor_spacing: float = 1000.0
    move_factor: float = 1.0
    diagonal_threshold: float = 0.5
    hand_orientation: HandOrientation = HandOrientation.NOT_MIRRORED
    stiffness_value_normalized = 1

    # Pattern controls
    outward_fraction_of_half_width: float = 0.5
    # line_steps: int = 40
    # circle_steps: int = 220
    # return_steps: int = 40
    line_steps: int = 40
    circle_steps: int = 220
    return_steps: int = 40

    oppose_object_motion: bool = True
    # Analysis controls.
    # The 2-D trilateration residual flags wire/geometry inconsistencies. All four
    # strategies now share the SAME isosceles mechanism triangle (default_model) for
    # both encode and decode, so the commanded path reconstructs round (no ellipse).
    # The 2-D strategies (cardinal, cardinal_diagonal, free_form) sit at residual
    # ~0.16-0.6. The IK strategy carries an extra ~1.4 (max ~3.2) BASELINE residual
    # that is NOT a wire fault: it is the genuine 3-D working-height signature (the
    # tactor sits at z=6 IK units above the anchor plane, so IK wire lengths are 3-D
    # while reconstruction is 2-D). At 1.25 this baseline trips ~203 IK steps by
    # design, keeping the residual visible in the plots/counts. Raise to ~3.3 for a
    # strategy-fair view that suppresses the IK working-height baseline. See the
    # markdown note in the "IK MovementStrategy comparison" section below.
    wire_residual_alert_threshold: float = 1.25

    # Live display controls
    live_plots: bool = False
    live_update_interval: int = 8


config = SimulationConfig()
config


In [ ]:
DEFAULT_PATTERN_TEMPLATE: list[dict[str, Any]] = [
    {
        'type': 'line',
        'name': 'outward_right',
        'start': (0.0, 0.0),
        'end': '$OUTWARD_POINT',
        'steps': 'line_steps',
    },
    {
        'type': 'circle',
        'name': 'clockwise_circle',
        'center': (0.0, 0.0),
        'radius': '$OUTWARD_RADIUS',
        'start_angle_deg': 0.0,
        'sweep_angle_deg': -360.0,
        'steps': 'circle_steps',
    },
    {
        'type': 'line',
        'name': 'back_to_center',
        'start': '$OUTWARD_POINT',
        'end': (0.0, 0.0),
        'steps': 'return_steps',
    },
]


def _resolve_value(value: Any, context: dict[str, Any], cfg: SimulationConfig) -> Any:
    if isinstance(value, str):
        if value.startswith('$'):
            return context[value]
        if hasattr(cfg, value):
            return getattr(cfg, value)
    return value


def build_pattern_points(
    cfg: SimulationConfig,
    pattern_template: list[dict[str, Any]] | None = None,
) -> pd.DataFrame:
    template = pattern_template or DEFAULT_PATTERN_TEMPLATE

    half_width = cfg.top_width / 2.0
    outward_x = half_width * cfg.outward_fraction_of_half_width
    context = {
        '$OUTWARD_POINT': (outward_x, 0.0),
        '$OUTWARD_RADIUS': outward_x,
    }

    rows: list[dict[str, float | str]] = []
    include_start = True

    for segment in template:
        segment_type = str(segment['type'])
        segment_name = str(segment.get('name', segment_type))
        steps = int(_resolve_value(segment['steps'], context, cfg))
        if steps < 1:
            raise ValueError(f'Invalid steps for {segment_name}: {steps}')

        if segment_type == 'line':
            start = np.array(_resolve_value(segment['start'], context, cfg), dtype=float)
            end = np.array(_resolve_value(segment['end'], context, cfg), dtype=float)

            t_values = np.linspace(0.0, 1.0, num=steps + 1)
            if not include_start:
                t_values = t_values[1:]

            for t in t_values:
                point = start + t * (end - start)
                rows.append({'segment': segment_name, 'x': float(point[0]), 'y': float(point[1])})

        elif segment_type == 'circle':
            center = np.array(_resolve_value(segment['center'], context, cfg), dtype=float)
            radius = float(_resolve_value(segment['radius'], context, cfg))
            start_angle = math.radians(float(_resolve_value(segment.get('start_angle_deg', 0.0), context, cfg)))
            sweep_angle = math.radians(float(_resolve_value(segment.get('sweep_angle_deg', -360.0), context, cfg)))

            angles = np.linspace(start_angle, start_angle + sweep_angle, num=steps + 1)
            if not include_start:
                angles = angles[1:]

            for angle in angles:
                x = center[0] + radius * math.cos(angle)
                y = center[1] + radius * math.sin(angle)
                rows.append({'segment': segment_name, 'x': float(x), 'y': float(y)})

        else:
            raise ValueError(f'Unsupported segment type: {segment_type}')

        include_start = False

    path_df = pd.DataFrame(rows)
    path_df.insert(0, 'step', np.arange(len(path_df), dtype=int))
    return path_df


pattern_df = build_pattern_points(config)

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(pattern_df['x'], pattern_df['y'], linewidth=1.5)
ax.scatter(pattern_df.iloc[0]['x'], pattern_df.iloc[0]['y'], color='green', label='Start', zorder=3)
ax.scatter(pattern_df.iloc[-1]['x'], pattern_df.iloc[-1]['y'], color='red', label='End', zorder=3)
ax.set_title('Configured movement pattern')
ax.set_xlabel('Requested X')
ax.set_ylabel('Requested Y')
ax.set_aspect('equal', 'box')
ax.legend()
plt.show()

pattern_df.head()


In [ ]:
from kinematics import unified_ik_starter as ik_module


def get_motor_anchors(motor_spacing: float) -> np.ndarray:
    """Reconstruction anchors matching the *real* mechanism triangle.

    The physical mechanism solved by ``unified_ik_starter`` is the isosceles
    triangle defined in ``default_model()`` (base 24.2, height 25.9 - NOT
    equilateral). Motor commands are wire-length deltas measured against that
    triangle and scaled by ``motor_spacing / base_span``. To reconstruct the
    tactor consistently for *every* strategy, the decode must use the SAME
    triangle scaled by the SAME factor; otherwise a commanded circle is read
    back as an ellipse and trilateration leaves a large residual.

    Motor order matches the IK leg order: 0=top, 1=bottom-right, 2=bottom-left.
    """
    anchors = ik_module.default_model()["anchors"]
    base_span = math.hypot(
        anchors["right"][0] - anchors["left"][0],
        anchors["right"][1] - anchors["left"][1],
    )
    scale = motor_spacing / base_span
    return np.array(
        [anchors[leg][:2] for leg in ("top", "right", "left")],
        dtype=float,
    ) * scale


def reconstruct_tactor_xy(
    final_lengths: np.ndarray,
    anchors: np.ndarray,
) -> tuple[float, float, float, np.ndarray, np.ndarray]:
    # Linearized trilateration from three anchor distances
    p1 = anchors[0]
    r1 = final_lengths[0]

    A = []
    b = []
    for i in (1, 2):
        xi, yi = anchors[i]
        ri = final_lengths[i]
        A.append([2.0 * (xi - p1[0]), 2.0 * (yi - p1[1])])
        b.append(
            (r1 ** 2 - ri ** 2)
            - (p1[0] ** 2 - xi ** 2)
            - (p1[1] ** 2 - yi ** 2)
        )

    A = np.asarray(A, dtype=float)
    b = np.asarray(b, dtype=float)
    solution, *_ = np.linalg.lstsq(A, b, rcond=None)
    x = float(solution[0])
    y = float(solution[1])

    predicted_lengths = np.linalg.norm(anchors - np.array([x, y]), axis=1)
    length_error = predicted_lengths - final_lengths
    residual_rmse = float(np.sqrt(np.mean(length_error ** 2)))

    return x, y, residual_rmse, predicted_lengths, length_error

In [ ]:
def show_live_state(partial_df: pd.DataFrame, strategy_name: str, alert_threshold: float) -> None:
    clear_output(wait=True)

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    idx0 = int(partial_df['motor_index_0'].iloc[0])
    idx1 = int(partial_df['motor_index_1'].iloc[0])
    idx2 = int(partial_df['motor_index_2'].iloc[0])

    axes[0].plot(partial_df['step'], partial_df['motor_0'], label=f'Motor {idx0}')
    axes[0].plot(partial_df['step'], partial_df['motor_1'], label=f'Motor {idx1}')
    axes[0].plot(partial_df['step'], partial_df['motor_2'], label=f'Motor {idx2}')
    axes[0].set_title(f'{strategy_name}: motor commands')
    axes[0].set_xlabel('Step')
    axes[0].set_ylabel('Motor position command')
    axes[0].legend()

    axes[1].plot(
        partial_df['object_x'],
        partial_df['object_y'],
        '--',
        label='Virtual object path',
        alpha=0.8,
        zorder=1,
    )
    axes[1].plot(
        partial_df['tactor_x'],
        partial_df['tactor_y'],
        label='Reconstructed tactor',
        linewidth=2,
        zorder=2,
    )

    issues = partial_df[partial_df['wire_issue']]
    if not issues.empty:
        axes[1].scatter(
            issues['tactor_x'],
            issues['tactor_y'],
            c='red',
            s=42,
            marker='o',
            edgecolors='black',
            linewidths=0.8,
            label='Wire issue',
            zorder=10,
        )

    axes[1].set_title(f'{strategy_name}: virtual object vs reconstructed tactor')
    axes[1].set_xlabel('X')
    axes[1].set_ylabel('Y')
    axes[1].set_aspect('equal', 'box')
    axes[1].legend(loc='upper right')

    fig.suptitle(f'Live update | issue threshold={alert_threshold:.2f}', y=1.03)
    plt.tight_layout()
    display(fig)
    plt.close(fig)


def simulate_strategy(
    strategy: MovementStrategy,
    pattern: pd.DataFrame,
    cfg: SimulationConfig,
) -> pd.DataFrame:
    controller = MotorController(
        movement_strategy=strategy,
        top_width=cfg.top_width,
        top_height=cfg.top_height,
        edge_threshold=cfg.edge_threshold,
        motor_spacing=cfg.motor_spacing,
        move_factor=cfg.move_factor,
        diagonal_threshold=cfg.diagonal_threshold,
        hand_orientation=cfg.hand_orientation,
    )

    motor_indices = [0, 1, 2]
    current_motor_positions = {idx: 0 for idx in motor_indices}

    anchors = get_motor_anchors(controller._motor_spacing)
    initial_lengths = np.linalg.norm(anchors - np.array([0.0, 0.0]), axis=1)

    rows: list[dict[str, Any]] = []

    final_step = int(pattern['step'].iat[-1])
    update_every = max(1, cfg.live_update_interval)

    for _, point in pattern.iterrows():
        step = int(point['step'])
        object_x = float(point['x'])
        object_y = float(point['y'])
        segment = str(point['segment'])

        target_x, target_y = map_object_displacement_to_tactor(
            obj_x=object_x,
            obj_y=object_y,
            oppose_motion=cfg.oppose_object_motion,
        )

        movements = controller.calculate_motor_movements(
            motor_set_id=MotorSetId.MOTORS_0_2,
            stiffness_value=cfg.stiffness_value_normalized,
            obj_x=target_x,
            obj_y=target_y,
            motors_enabled=True,
            reset_to_origin=False,
        )

        for movement in movements:
            if movement.index in current_motor_positions:
                current_motor_positions[movement.index] = movement.pos

        motor_offsets = np.array([current_motor_positions[idx] for idx in motor_indices], dtype=float)
        final_lengths = initial_lengths + motor_offsets

        wire_too_tight = bool(np.any(final_lengths <= 0.0))
        if wire_too_tight:
            tactor_x = float('nan')
            tactor_y = float('nan')
            residual_rmse = float('inf')
            length_error = np.array([float('nan'), float('nan'), float('nan')], dtype=float)
            dominant_wire_motor_index = -1
            dominant_wire_error = float('nan')
        else:
            tactor_x, tactor_y, residual_rmse, _, length_error = reconstruct_tactor_xy(final_lengths, anchors)
            dominant_wire_idx = int(np.argmax(np.abs(length_error)))
            dominant_wire_motor_index = motor_indices[dominant_wire_idx]
            dominant_wire_error = float(abs(length_error[dominant_wire_idx]))

        tactor_error = float('nan')
        if not math.isnan(tactor_x) and not math.isnan(tactor_y):
            tactor_error = float(math.hypot(tactor_x - target_x, tactor_y - target_y))

        wire_issue = wire_too_tight or (residual_rmse > cfg.wire_residual_alert_threshold)

        rows.append(
            {
                'strategy': strategy.value,
                'step': step,
                'segment': segment,
                'object_x': object_x,
                'object_y': object_y,
                'target_x': target_x,
                'target_y': target_y,
                'motor_index_0': motor_indices[0],
                'motor_index_1': motor_indices[1],
                'motor_index_2': motor_indices[2],
                'motor_0': float(motor_offsets[0]),
                'motor_1': float(motor_offsets[1]),
                'motor_2': float(motor_offsets[2]),
                'wire_length_0': float(final_lengths[0]),
                'wire_length_1': float(final_lengths[1]),
                'wire_length_2': float(final_lengths[2]),
                'tactor_x': tactor_x,
                'tactor_y': tactor_y,
                'tactor_error': tactor_error,
                'wire_residual_rmse': float(residual_rmse),
                'wire_too_tight': wire_too_tight,
                'wire_issue': wire_issue,
                'dominant_wire_motor_index': dominant_wire_motor_index,
                'dominant_wire_error': dominant_wire_error,
            }
        )

        if cfg.live_plots and (step % update_every == 0 or step == final_step):
            show_live_state(pd.DataFrame(rows), strategy.value, cfg.wire_residual_alert_threshold)

    return pd.DataFrame(rows)


In [ ]:
strategies = [
    MovementStrategy.CARDINAL,
    MovementStrategy.CARDINAL_DIAGONAL,
    MovementStrategy.FREE_FORM,
    MovementStrategy.IK,
]

pattern_df = build_pattern_points(config)
results = [simulate_strategy(strategy, pattern_df, config) for strategy in strategies]
all_results = pd.concat(results, ignore_index=True)

print(f'Path points: {len(pattern_df)}')
print(f'Strategies: {len(strategies)}')
print(f'Total rows: {len(all_results)}')
all_results.head()


In [ ]:
def evenly_sample_steps(part: pd.DataFrame, sample_count: int) -> pd.DataFrame:
    if sample_count < 2:
        raise ValueError('sample_count must be at least 2.')

    if part.empty:
        return part

    sampled_len = min(sample_count, len(part))
    sampled_indices = np.linspace(0, len(part) - 1, num=sampled_len, dtype=int)
    sampled_indices = np.unique(sampled_indices)
    return part.iloc[sampled_indices]

def plot_motor_movements(df: pd.DataFrame, sample_count: int | None) -> None:
    strategy_names = [s.value for s in strategies]
    motor_colors = {
        0: 'lightskyblue',
        1: 'blue',
        2: 'purple',
    }
    fig, axes = plt.subplots(len(strategy_names), 1, figsize=(14, 4 * len(strategy_names)), sharex=True)
    if len(strategy_names) == 1:
        axes = [axes]

    for i, strategy_name in enumerate(strategy_names):
        ax = axes[i]
        part = df[df['strategy'] == strategy_name].reset_index(drop=True)
        if sample_count:
            part = evenly_sample_steps(part, sample_count)
        idx0 = int(part['motor_index_0'].iloc[0])
        idx1 = int(part['motor_index_1'].iloc[0])
        idx2 = int(part['motor_index_2'].iloc[0])

        ax.plot(part['step'], part['motor_0'], '-o', label=f'Motor {idx0}', color=motor_colors[0])
        ax.plot(part['step'], part['motor_1'], '-o', label=f'Motor {idx1}', color=motor_colors[1])
        ax.plot(part['step'], part['motor_2'], '-o', label=f'Motor {idx2}', color=motor_colors[2])
        ax.set_title(f'{strategy_name}: motor command trace' + (f' ({len(part)} samples)' if sample_count else ''))
        ax.set_ylabel('Motor position command')
        ax.legend()

    axes[-1].set_xlabel('Step')
    plt.tight_layout()
    plt.show()

def plot_tactor_paths(df: pd.DataFrame, sample_count: int | None) -> None:
    strategy_names = [s.value for s in strategies]
    fig, axes = plt.subplots(1, len(strategy_names), figsize=(6 * len(strategy_names), 5), sharex=True, sharey=True)
    if len(strategy_names) == 1:
        axes = [axes]

    for i, strategy_name in enumerate(strategy_names):
        ax = axes[i]
        part = df[df['strategy'] == strategy_name].reset_index(drop=True)
        if sample_count:
            part = evenly_sample_steps(part, sample_count)

        ax.plot(
            part['object_x'],
            part['object_y'],
            '--o',
            label='Virtual object',
            zorder=1,
        )
        ax.plot(
            part['tactor_x'],
            part['tactor_y'],
            '-o',
            label='Reconstructed tactor',
            zorder=2,
        )

        issues = part[part['wire_issue']]
        if not issues.empty:
            ax.scatter(
                issues['tactor_x'],
                issues['tactor_y'],
                c='red',
                s=42,
                marker='o',
                edgecolors='black',
                linewidths=0.8,
                label='Wire issue',
                zorder=10,
            )

        ax.set_title(f'{strategy_name}' + (f' ({len(part)} samples)' if sample_count else ''))
        ax.set_xlabel('X')
        ax.set_ylabel('Y')
        ax.set_aspect('equal', 'box')
        ax.legend(loc='upper right')

    plt.suptitle('Virtual object vs reconstructed tactor trajectory', y=1.03)
    plt.tight_layout()
    plt.show()


def plot_error_metrics(df: pd.DataFrame, alert_threshold: float) -> None:
    strategy_names = [s.value for s in strategies]
    fig, axes = plt.subplots(len(strategy_names), 2, figsize=(15, 4 * len(strategy_names)), sharex=True)
    if len(strategy_names) == 1:
        axes = np.array([axes])

    for i, strategy_name in enumerate(strategy_names):
        part = df[df['strategy'] == strategy_name]

        axes[i, 0].plot(part['step'], part['tactor_error'])
        axes[i, 0].set_title(f'{strategy_name}: tactor target error')
        axes[i, 0].set_ylabel('Euclidean error')

        axes[i, 1].plot(part['step'], part['wire_residual_rmse'])
        axes[i, 1].axhline(alert_threshold, color='red', linestyle='--', linewidth=1, label='Issue threshold')
        axes[i, 1].set_title(f'{strategy_name}: wire residual RMSE')
        axes[i, 1].legend()

    axes[-1, 0].set_xlabel('Step')
    axes[-1, 1].set_xlabel('Step')
    plt.tight_layout()
    plt.show()


plot_motor_movements(all_results, sample_count=0)
plot_tactor_paths(all_results, sample_count=0)
plot_error_metrics(all_results, config.wire_residual_alert_threshold)

summary_source = all_results.replace([np.inf, -np.inf], np.nan)
summary = (
    summary_source.groupby('strategy')
    .agg(
        mean_tactor_error=('tactor_error', 'mean'),
        max_tactor_error=('tactor_error', 'max'),
        mean_wire_residual=('wire_residual_rmse', 'mean'),
        max_wire_residual=('wire_residual_rmse', 'max'),
        issue_steps=('wire_issue', 'sum'),
        issue_ratio=('wire_issue', 'mean'),
    )
    .sort_index()
)

print('Strategy summary:')
display(summary)

issues = all_results[all_results['wire_issue']].copy()
if issues.empty:
    print('No wire consistency issues detected at the configured threshold.')
else:
    print(f'Wire issues detected in {len(issues)} steps.')
    issue_cols = [
        'strategy',
        'step',
        'segment',
        'target_x',
        'target_y',
        'tactor_x',
        'tactor_y',
        'wire_too_tight',
        'wire_residual_rmse',
        'dominant_wire_motor_index',
        'dominant_wire_error',
    ]
    display(issues[issue_cols].head(100))


## IK MovementStrategy comparison

The IK path now runs through the same in-repository `MovementStrategy.IK` implementation used by `backend.py`.
There are no outside-repo path imports or notebook-local IK actuator calculations here; the notebook compares the public motor-command output produced by `MotorController.calculate_motor_movements(...)` for each strategy.


### Reading the IK wire-residual count

All four strategies now share the **same isosceles mechanism triangle**
(`unified_ik_starter.default_model()`, scaled by `motor_spacing / base_span`) for
*both* the motor-command encode (`MotorController._calculate_movements_to_point` /
`_calculate_ik_tactor_wire_length`) and the XY decode (`get_motor_anchors`). As a
result the commanded circle reconstructs **round** for every strategy (no ellipse),
and the three 2-D strategies — `cardinal`, `cardinal_diagonal`, `free_form` —
report **0 wire issues** (residual ≈ 0.16–0.6).

The `ik` strategy still flags ~203 steps, but this is **not** a wire fault. It is the
genuine **3-D working-height signature**: the IK solver places the tactor at
`z = 6` IK units above the anchor plane (required for valid leg solutions), so IK's
wire lengths are 3-D while the reconstruction is 2-D. This leaves a uniform baseline
residual of ~1.4 (max ~3.2) — above the 1.25 alert threshold, but far below the
tens-of-units residuals a real geometry/wire inconsistency produces (e.g. the ~40
seen when the encode and decode triangles disagree).

In short: **the shape is correct for all four strategies; the IK residual is the
expected planar projection of its working height.** Raise
`wire_residual_alert_threshold` to ~3.3 if you prefer a strategy-fair view that
suppresses this baseline.


In [ ]:
def summarize_strategy_results(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for strategy in strategies:
        part = df[df['strategy'] == strategy.value]
        rows.append(
            {
                'strategy': strategy.value,
                'rows': len(part),
                'wire_issue_count': int(part['wire_issue'].sum()),
                'max_tactor_error': float(part['tactor_error'].max(skipna=True)),
                'max_wire_residual_rmse': float(part['wire_residual_rmse'].replace(np.inf, np.nan).max(skipna=True)),
                'motor_0_min': float(part['motor_0'].min(skipna=True)),
                'motor_0_max': float(part['motor_0'].max(skipna=True)),
                'motor_1_min': float(part['motor_1'].min(skipna=True)),
                'motor_1_max': float(part['motor_1'].max(skipna=True)),
                'motor_2_min': float(part['motor_2'].min(skipna=True)),
                'motor_2_max': float(part['motor_2'].max(skipna=True)),
            }
        )
    return pd.DataFrame(rows)


ik_results = all_results[all_results['strategy'] == MovementStrategy.IK.value].reset_index(drop=True)
if ik_results.empty:
    raise RuntimeError('MovementStrategy.IK was not included in strategies or produced no rows.')

ik_controller = MotorController(
    movement_strategy=MovementStrategy.IK,
    top_width=config.top_width,
    top_height=config.top_height,
    edge_threshold=config.edge_threshold,
    motor_spacing=config.motor_spacing,
    move_factor=config.move_factor,
    diagonal_threshold=config.diagonal_threshold,
    hand_orientation=config.hand_orientation,
)
ik_solver_path = Path(ik_controller._get_ik_module().__file__).resolve()

print('IK strategy is simulated through MotorController(MovementStrategy.IK).')
print(f'IK solver path: {ik_solver_path.relative_to(REPO_ROOT)}')
print(
    'Backend-equivalent inputs: oppose_object_motion=' 
    f'{config.oppose_object_motion}, stiffness={config.stiffness_value_normalized}, '
    f'move_factor={config.move_factor}, motor_spacing={config.motor_spacing}.'
)

display(summarize_strategy_results(all_results))
print('IK sample rows:')
display(
    ik_results[
        [
            'step', 'segment', 'object_x', 'object_y', 'target_x', 'target_y',
            'motor_index_0', 'motor_0', 'motor_index_1', 'motor_1', 'motor_index_2', 'motor_2',
            'wire_issue', 'tactor_error', 'wire_residual_rmse',
        ]
    ].head(10)
)
